# Sandbox: Step-by-Step Production Pipeline

Educational Goal:
- Understand how the production pipeline works
- Execute each stage interactively
- Debug modules without editing src/

This notebook mirrors src/main.py intentionally.
Do NOT rewrite logic here.
Only orchestrate existing modules.

In [56]:
import sys
from pathlib import Path

# Find project root (one level up from notebooks)
PROJECT_ROOT = Path().resolve().parents[0]

# Add root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/nicopbeard/Documents/mlops/mlops-group8


In [ ]:
import pandas as pd  # type: ignore
from sklearn.model_selection import train_test_split  # type: ignore

from src.utils import setup_logging
import src.load_data as load_data
import src.clean_data as clean_data
import src.validate as validate
import src.features as features
import src.train as train
import src.evaluate as evaluate
import src.infer as infer
import yaml  # type: ignore

from src.utils import save_csv, save_json, save_model


In [58]:
def load_config(config_path: str):
    with open(config_path, 'r') as f:
        return yaml.safe_load(f)
    
SETTINGS = load_config(f"{PROJECT_ROOT}/config.yaml")

setup_logging(
    level=SETTINGS.get("logging", {}).get("level", "INFO"),
    log_file=SETTINGS.get("logging", {}).get("file"),
)

In [59]:
print("Using SETTINGS:", SETTINGS)

Using SETTINGS: {'project': {'name': 'Spotify Sound Archetypes', 'problem_type': 'regression', 'target_column': 'popularity'}, 'paths': {'raw_data': 'data/raw/SpotifyAudioFeaturesApril2019.csv', 'clean_data': 'data/processed/clean.csv', 'model_path': 'models/model.joblib', 'report_path': 'reports/predictions.csv'}, 'train': {'test_size': 0.2, 'val_size': 0.2, 'seed': 42, 'rf_n_estimators': 100, 'rf_max_depth': 10}, 'features': {'quantile_bin': ['duration_ms', 'tempo'], 'categorical_onehot': ['key', 'mode'], 'numeric_passthrough': ['acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'valence'], 'n_bins': 5}}


Infrastructure Setup

In [60]:
for folder in ["data/raw", "data/processed", "models", "reports"]:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

print("Infrastructure folders ensured.")

Infrastructure folders ensured.


Load Data

In [61]:
raw_path = PROJECT_ROOT / SETTINGS["paths"]["raw_data"]
df_raw = load_data.load_raw_data(raw_path)

print("Raw shape:", df_raw.shape)
df_raw.head()

2026-03-04 21:09:15 | INFO | src.load_data | Attempting to load raw data from /Users/nicopbeard/Documents/mlops/mlops-group8/data/raw/SpotifyAudioFeaturesApril2019.csv
2026-03-04 21:09:15 | INFO | src.utils | Reading CSV from /Users/nicopbeard/Documents/mlops/mlops-group8/data/raw/SpotifyAudioFeaturesApril2019.csv
2026-03-04 21:09:15 | INFO | src.load_data | Raw data loaded successfully. shape=(130663, 17)


Raw shape: (130663, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


Clean Data

In [62]:
df_clean = clean_data.clean_dataframe(df_raw, SETTINGS["project"]["target_column"])

save_csv(df_clean, PROJECT_ROOT / SETTINGS["paths"]["clean_data"])

print("Clean shape:", df_clean.shape)
df_clean.head()

2026-03-04 21:09:16 | INFO | src.clean_data | Cleaning dataframe...
2026-03-04 21:09:16 | INFO | src.utils | Saving CSV to /Users/nicopbeard/Documents/mlops/mlops-group8/data/processed/clean.csv


Clean shape: (129858, 17)


,artist_name,track_id,track_name,acousticness,danceability,duration_ms,energy,instrumentalness,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,popularity
0,YG,2RM4jf1Xa9zPgMGRDiht8O,"Big Bank feat. 2 Chainz, Big Sean, Nicki Minaj",0.005820,0.743,238373,0.339,0.000,1,0.0812,-7.678,1,0.4090,203.927,4,0.118,15
1,YG,1tHDG53xJNGsItRA3vfVgs,BAND DRUM (feat. A$AP Rocky),0.024400,0.846,214800,0.557,0.000,8,0.2860,-7.259,1,0.4570,159.009,4,0.371,0
2,R3HAB,6Wosx2euFPMT14UXiWudMy,Radio Silence,0.025000,0.603,138913,0.723,0.000,9,0.0824,-5.890,0,0.0454,114.966,4,0.382,56
3,Chris Cooq,3J2Jpw61sO7l6Hc7qdYV91,Lactose,0.029400,0.800,125381,0.579,0.912,5,0.0994,-12.118,0,0.0701,123.003,4,0.641,0
4,Chris Cooq,2jbYvQCyPgX3CdmAzeVeuS,Same - Original mix,0.000035,0.783,124016,0.792,0.878,7,0.0332,-10.277,1,0.0661,120.047,4,0.928,0


Validate

In [63]:
required_cols = (
        SETTINGS["features"]["quantile_bin"] + 
        SETTINGS["features"]["categorical_onehot"] + 
        SETTINGS["features"]["numeric_passthrough"] + 
        [SETTINGS["project"]["target_column"]]
    )
validate.validate_dataframe(
        df_clean,
        required_cols,
        target_column=SETTINGS["project"]["target_column"],
        allow_feature_nulls=True
    )
print("Validation passed.")

2026-03-04 21:09:16 | INFO | src.validate | Validating data schema...
2026-03-04 21:09:16 | INFO | src.validate | Schema and null-check validation passed.


Validation passed.


Train/Test Split

In [64]:
target = SETTINGS["project"]["target_column"]
X = df_clean.drop(columns=[target])
y = df_clean[target]

test_size = SETTINGS["train"]["test_size"]
val_size = SETTINGS["train"].get("val_size", 0.2)
seed = SETTINGS["train"]["seed"]

# Stratify only for classification
strat = y if SETTINGS["project"]["problem_type"] == "classification" else None

# 5.1 Train+Val vs Test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=seed,
    stratify=strat,
)

# 5.2 Train vs Val (val_size is fraction of trainval)
strat_trainval = (
    y_trainval if SETTINGS["project"]["problem_type"] == "classification" else None
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=val_size,
    random_state=seed,
    stratify=strat_trainval,
)

print(
    f"Split sizes: train={len(X_train)} ({100*len(X_train)/len(X):.1f}%), "
    f"val={len(X_val)} ({100*len(X_val)/len(X):.1f}%), "
    f"test={len(X_test)} ({100*len(X_test)/len(X):.1f}%)"
)

Split sizes: train=83108 (64.0%), val=20778 (16.0%), test=25972 (20.0%)


Feature Processor

In [65]:
preprocessor = features.get_feature_preprocessor(
        quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
        categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
        numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
        n_bins=SETTINGS["features"]["n_bins"]
    )

print("Feature recipe built.")

2026-03-04 21:09:16 | INFO | src.features | Building feature preprocessor recipe...


Feature recipe built.


Train Model

In [66]:
model_pipeline = train.train_model(
    X_train,
    y_train,
    preprocessor,
    SETTINGS["project"]["problem_type"],
    train_config=SETTINGS.get("train", {}),
)

print("Initial model trained (train split). Not saved yet (matches src.main).")

2026-03-04 21:09:16 | INFO | src.train | Training regression model...


Initial model trained (train split). Not saved yet (matches src.main).


Evaluate

In [67]:
# 8. Evaluate on Val
val_metrics = evaluate.evaluate_model(
    model_pipeline,
    X_val,
    y_val,
    SETTINGS["project"]["problem_type"],
)

# Rebuild preprocessor fresh (src.main does this)
final_preprocessor = features.get_feature_preprocessor(
    quantile_bin_cols=SETTINGS["features"]["quantile_bin"],
    categorical_onehot_cols=SETTINGS["features"]["categorical_onehot"],
    numeric_passthrough_cols=SETTINGS["features"]["numeric_passthrough"],
    n_bins=SETTINGS["features"]["n_bins"],
)

# Retrain final model on Train+Val and save
final_model_pipeline = train.train_model(
    X_trainval,
    y_trainval,
    final_preprocessor,
    SETTINGS["project"]["problem_type"],
    train_config=SETTINGS.get("train", {}),
)

save_model(final_model_pipeline, PROJECT_ROOT / SETTINGS["paths"]["model_path"])

# Evaluate final model on Test
test_metrics = evaluate.evaluate_model(
    final_model_pipeline,
    X_test,
    y_test,
    SETTINGS["project"]["problem_type"],
)

metrics = {"val": val_metrics, "test": test_metrics}

save_json(metrics, PROJECT_ROOT / "reports" / "metrics.json")
save_json(SETTINGS, PROJECT_ROOT / "reports" / "run_config.json")

print("Validation metrics:", val_metrics)
print("Test metrics:", test_metrics)

2026-03-04 21:09:19 | INFO | src.evaluate | Evaluating model performance...
2026-03-04 21:09:19 | INFO | src.evaluate | Metrics: rmse=18.237008 mae=14.914996
2026-03-04 21:09:19 | INFO | src.features | Building feature preprocessor recipe...
2026-03-04 21:09:19 | INFO | src.train | Training regression model...
2026-03-04 21:09:24 | INFO | src.utils | Saving model to /Users/nicopbeard/Documents/mlops/mlops-group8/models/model.joblib
2026-03-04 21:09:24 | INFO | src.evaluate | Evaluating model performance...
2026-03-04 21:09:24 | INFO | src.evaluate | Metrics: rmse=18.146484 mae=14.823458
2026-03-04 21:09:24 | INFO | src.utils | Saving JSON to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/metrics.json
2026-03-04 21:09:24 | INFO | src.utils | Saving JSON to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/run_config.json


Validation metrics: {'rmse': 18.237008290824583, 'mae': 14.914995926219108}
Test metrics: {'rmse': 18.146484220001206, 'mae': 14.823457601968261}


Inference

In [68]:
df_preds = infer.run_inference(final_model_pipeline, X_test)
save_csv(df_preds, PROJECT_ROOT / SETTINGS["paths"]["report_path"])

df_preds.head()

2026-03-04 21:09:24 | INFO | src.infer | Running inference on new data...
2026-03-04 21:09:24 | INFO | src.infer | Inference complete. Generated 25972 predictions.
2026-03-04 21:09:24 | INFO | src.utils | Saving CSV to /Users/nicopbeard/Documents/mlops/mlops-group8/reports/predictions.csv


,prediction
0,35.803730
1,25.457876
2,25.059417
3,16.995746
4,27.028266
